In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

import re
import numpy as np
import pandas as pd


from inDecay import PATH,my_utils
os.chdir(PATH.main_dir)
from inDecay.models import Topk_Event_Overlapping

In [2]:

import pickle as pkl
from scipy.stats import pearsonr

pj = os.path.join

In [3]:
from qrguide import analysis_fn, transformation

In [4]:
## read Test set 
experiments =["ST_June_2017_BOB_LV7A_DPI7", "ST_June_2017_BOB_LV7B_DPI7" ,
      "ST_June_2017_CHO_LV7A_DPI7", "ST_June_2017_CHO_LV7B_DPI7",
      "ST_June_2017_E14TG2A_LV7A_DPI7", "ST_June_2017_E14TG2A_LV7B_DPI7",
      "ST_June_2017_HAP1_LV7A_DPI7", "ST_June_2017_HAP1_LV7B_DPI7",
      "ST_June_2017_K562_800x_LV7A_DPI7", "ST_June_2017_K562_800x_LV7B_DPI7",]

celltypes = [exp.split("_")[3] for exp in experiments if 'LV7A' in exp]
celltype_rename = ['iPSC', 'CHO', 'mESC', 'HAP1', 'K562']

In [5]:
celltypes, celltype_rename

(['BOB', 'CHO', 'E14TG2A', 'HAP1', 'K562'],
 ['iPSC', 'CHO', 'mESC', 'HAP1', 'K562'])

In [6]:
def read_pkl(path):
    with open(path, 'rb') as f:
        Y = pkl.load(f)
    f.close()
    return Y

def evalute_fn(Y_true_path, Y_pred_path):
    Y_pred =read_pkl(Y_pred_path)
    Y = read_pkl(Y_true_path)

    eval_json = analysis_fn.assessment_recipe_forecast(Y, Y_pred)
    eval_json.update(analysis_fn.assessment_recipe_IDL_forecast(Y, Y_pred))

    eval_df = pd.json_normalize(eval_json)
    return eval_df

# Feat v5

In [9]:
Y_true_path_dict = {
    "iPSC":"pl_trainer_log/ST_featv5fast_ST_DeepDecay_identity_C500/ST_June_2017_BOB_LV7A_DPI7/ForeCast_TestY.pkl",
    "CHO" :"pl_trainer_log/ST_featv5fast_ST_DeepDecay_identity_C500/ST_June_2017_CHO_LV7A_DPI7/ForeCast_TestY.pkl",
    "mESC":"pl_trainer_log/ST_featv5fast_ST_DeepDecay_identity_C500/ST_June_2017_E14TG2A_LV7A_DPI7/ForeCast_TestY.pkl",
    "HAP1":"pl_trainer_log/ST_featv5fast_ST_DeepDecay_identity_C500/ST_June_2017_HAP1_LV7A_DPI7/ForeCast_TestY.pkl",
    "K562":"pl_trainer_log/ST_featv5fast_ST_DeepDecay_identity_C500/ST_June_2017_K562_LV7A_DPI7/ForeCast_TestY.pkl"
}

v5_pred_path_dict = {
    "iPSC": "/ssd/users/louisayu/inDecay/pretrained/iPSC_featv5_pretrainedTestPred.pkl",
    "mESC" : "/ssd/users/louisayu/inDecay/pretrained/mESC_featv5_pretrainedTestPred.pkl",
    "HAP1" : "/ssd/users/louisayu/inDecay/pretrained/HAP1_featv5_pretrainedTestPred.pkl",
    "CHO" : "/ssd/users/louisayu/inDecay/pretrained/CHO_featv5_pretrainedTestPred.pkl",
    "K562" : "/ssd/users/louisayu/inDecay/pretrained/K562_featv5_pretrainedTestPred.pkl"
}

In [10]:
v5_df = []
for cell in celltype_rename:
    df = evalute_fn(Y_true_path_dict[cell], v5_pred_path_dict[cell])
    df['celltype'] = cell
    df['model'] = 'featv5'

    v5_df.append(df)

# merge all celltype
featv5_df = pd.concat(v5_df, axis=0).reset_index().iloc[:, 1:]

# rename model as set
featv5_df.rename({"model":"set"}, axis=1, inplace=True)

In [12]:
featv5_df.head()

,KL divergence,Top5 events recall,Top10 events recall,R2 of Frameshift ratio,Coll_I_Top5,Coll_I_Top10,KLD_IDL,Top5_IDL,Top10_IDL,W1-distance_IDL,delratio_r2,Kendall_tau_IDL,celltype,set
0,2.055804,3.105914,5.813769,0.848340,3.217123,5.960282,0.238307,3.460724,7.011474,0.071170,0.786620,0.674188,iPSC,featv5
1,1.229231,3.347749,6.762577,0.803359,3.458076,6.991174,0.145973,3.696381,7.807590,0.056674,0.780138,0.769071,CHO,featv5
2,1.424817,3.279788,6.517211,0.859038,3.413945,6.664607,0.196741,3.654898,7.917034,0.067102,0.774725,0.745232,mESC,featv5
3,1.258304,3.282436,6.386584,0.849073,3.406002,6.552515,0.155019,3.601942,7.496911,0.056359,0.827923,0.763278,HAP1,featv5
4,1.125679,3.346867,6.563989,0.845975,3.407767,6.677846,0.150287,3.648720,7.607237,0.054633,0.847628,0.766408,K562,featv5


In [14]:
featv5_df.to_csv(f"{PATH.main_dir}/results/benchmarking/featv5_perform_Aug28.csv", index=False)

# Feat v3

In [ ]:
Y_true_path_dict = {
    "iPSC":"pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_BOB_LV7A_DPI7/ForeCast_TestY.pkl",
    "CHO" :"pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_CHO_LV7A_DPI7/ForeCast_TestY.pkl",
    "mESC":"pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_E14TG2A_LV7A_DPI7/ForeCast_TestY.pkl",
    "HAP1":"pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_HAP1_LV7A_DPI7/ForeCast_TestY.pkl",
    "K562":"pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_K562_LV7A_DPI7/ForeCast_TestY.pkl"
}

v3_pred_path_dict = {
    "iPSC": "pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_BOB_LV7A_DPI7/lightning_logs/version_2/checkpoints/epoch=96-val_cre=2.95276666TestPred.pkl",
    "CHO": "pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_CHO_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=54-val_cre=3.28731441TestPred.pkl",
    "mESC": "pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_E14TG2A_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=62-val_cre=3.04122829TestPred.pkl",
    "HAP1": "pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_HAP1_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=53-val_cre=3.25698686TestPred.pkl",
    "K562": "pl_trainer_log/ST_featv3_ST_DeepDecay_interaction/ST_June_2017_K562_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=90-val_cre=3.35240030TestPred.pkl"
}


In [26]:
v3_df = []
for cell in celltype_rename:
    df = evalute_fn(Y_true_path_dict[cell], v3_pred_path_dict[cell])
    df['celltype'] = cell
    df['model'] = 'featv3'

    v3_df.append(df)

In [29]:
# merge all celltype
featv3_df = pd.concat(v3_df, axis=0).reset_index().iloc[:, 1:]

# rename model as set
featv3_df.rename({"model":"set"}, axis=1, inplace=True)

# feat v4 

## interaction

In [9]:
Y_true_path_dict = {
    "iPSC":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_BOB_LV7A_DPI7/ForeCast_TestY.pkl",
    "CHO" :"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_CHO_LV7A_DPI7/ForeCast_TestY.pkl",
    "mESC":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_E14TG2A_LV7A_DPI7/ForeCast_TestY.pkl",
    "HAP1":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_HAP1_LV7A_DPI7/ForeCast_TestY.pkl",
    "K562":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_K562_LV7A_DPI7/ForeCast_TestY.pkl"
}

Y_pred_path_dict = {
    "iPSC": "pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_BOB_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=77-val_cre=2.94685531TestPred.pkl",
    "CHO":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_CHO_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=66-val_cre=3.27570724TestPred.pkl",
    "mESC":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_E14TG2A_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=93-val_cre=3.03215241TestPred.pkl",
    "HAP1":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_HAP1_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=50-val_cre=3.24646974TestPred.pkl",
    "K562":"pl_trainer_log/ST_featv4_ST_DeepDecay_interaction/ST_June_2017_K562_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=84-val_cre=3.34424281TestPred.pkl"
}


In [11]:
interaction_df = []
for cell in celltype_rename:
    df = evalute_fn(Y_true_path_dict[cell], Y_pred_path_dict[cell])
    df['celltype'] = cell
    df['model'] = 'Interaction'

    interaction_df.append(df)

## identity

In [10]:
Y_pred_ident_dict = {
    "iPSC": "pl_trainer_log/ST_featv4_ST_DeepDecay_identity/ST_June_2017_BOB_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=99-val_cre=2.95777583TestPred.pkl",
    "CHO":"pl_trainer_log/ST_featv4_ST_DeepDecay_identity/ST_June_2017_CHO_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=95-val_cre=3.28872061TestPred.pkl",
    "mESC":"pl_trainer_log/ST_featv4_ST_DeepDecay_identity/ST_June_2017_E14TG2A_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=95-val_cre=3.04344153TestPred.pkl",
    "HAP1":"pl_trainer_log/ST_featv4_ST_DeepDecay_identity/ST_June_2017_HAP1_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=98-val_cre=3.25877213TestPred.pkl",
    "K562":"pl_trainer_log/ST_featv4_ST_DeepDecay_identity/ST_June_2017_K562_LV7A_DPI7/lightning_logs/version_0/checkpoints/epoch=97-val_cre=3.35827565TestPred.pkl"
}

In [12]:
for cell in celltype_rename:
    df = evalute_fn(Y_true_path_dict[cell], Y_pred_ident_dict[cell])
    df['celltype'] = cell
    df['model'] = 'identity'

    interaction_df.append(df)

In [34]:
# merge all celltype
featv4_df = pd.concat(interaction_df, axis=0).reset_index().iloc[:, 1:]

# rename model as set
featv4_df.rename({"model":"set"}, axis=1, inplace=True)
featv4_df['set'] = featv4_df.set.map(
    {"Interaction":"featv4 interaction",
    "identity":"featv4 identity"}
)

# save feat v5